In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

In [287]:
RETRIEVAL_DIR = Path("../output/retrieval")
DATA_DIR = Path("../data/gold_data")

METHOD = "bm25_rerank" # "bm25" # # "bm25_rerank" # "dense" # "dense_instruct" # "dense_instruct_rerank" # "dense_rerank" # hybrid # "hybrid_instruct" # "hybrid_instruct_rerank" # "hybrid_rerank"

RESULTS_DIR = Path(f"../results/retrieval/{METHOD}")
RESULTS_DIR.mkdir(parents=False, exist_ok=True)

In [288]:
# Evaluate either with or w/o graphics
NO_GRAPHICS = False

if NO_GRAPHICS:
    gold_processed_df = pd.read_csv(DATA_DIR / "gold_standard_processed_no_graphics.csv")
else:
    gold_processed_df = pd.read_csv(DATA_DIR / "gold_standard_processed.csv")

In [289]:
print(f"Gold before filtering NA: {len(gold_processed_df)}")
gold_processed_df = gold_processed_df.dropna(subset=["page"])
print(f"Gold after filtering NA: {len(gold_processed_df)}")

Gold before filtering NA: 713
Gold after filtering NA: 643


In [290]:
# Display types of each gold page, e.g. {("Allianz_2022", 79.0): {"Table"}}
display_types_by_page = gold_processed_df.groupby(["report_id", "page"])["display_type"].agg(set).to_dict()
display_types_by_page

{('Allianz_2022', 79.0): {'Table'},
 ('Allianz_2022', 93.0): {'Graphic', 'Graphic/Table'},
 ('Daimler_2020', 63.0): {'Table'},
 ('Fresenius SE_2019', 126.0): {'Table'},
 ('acuity brands inc_2022', 99.0): {'Table'},
 ('acuity brands inc_2022', 101.0): {'Table', 'Table/Text', 'Text/Table'},
 ('addtech_2022', 73.0): {'Table'},
 ('aixtron_2020', 50.0): {'Illustration'},
 ('aixtron_2020', 51.0): {'Text'},
 ('allfunds group_2021', 125.0): {'Table'},
 ('allfunds group_2021', 142.0): {'Table'},
 ('americold realty inc_2022', 53.0): {'Table'},
 ('apollo commercial real estate fina_2019', 9.0): {'Table'},
 ('arteria networks corp_2019', 10.0): {'Table'},
 ('autoneum holding_2019', 27.0): {'Table'},
 ('balanced commercial property trust_2020', 34.0): {'Table'},
 ('bekaert (d) sa_2022', 272.0): {'Other'},
 ('bekaert (d) sa_2022', 276.0): {'Table'},
 ('blackberry ltd_2022', 21.0): {'Table'},
 ('blackberry ltd_2022', 22.0): {'Table'},
 ('blackline inc_2021', 21.0): {'Table'},
 ('cabot corp_2018', 22

In [291]:
gold_pages_by_report = gold_processed_df.groupby("report_id")["page"].agg(set).to_dict()
gold_pages_by_report

{'Allianz_2022': {79.0, 93.0},
 'Daimler_2020': {63.0},
 'Fresenius SE_2019': {126.0},
 'acuity brands inc_2022': {99.0, 101.0},
 'addtech_2022': {73.0},
 'aixtron_2020': {50.0, 51.0},
 'allfunds group_2021': {125.0, 142.0},
 'americold realty inc_2022': {53.0},
 'apollo commercial real estate fina_2019': {9.0},
 'arteria networks corp_2019': {10.0},
 'autoneum holding_2019': {27.0},
 'balanced commercial property trust_2020': {34.0},
 'bekaert (d) sa_2022': {272.0, 276.0},
 'blackberry ltd_2022': {21.0, 22.0},
 'blackline inc_2021': {21.0},
 'cabot corp_2018': {22.0},
 'cardinal energy ltd_2021': {19.0},
 'century communities inc_2019': {11.0},
 'chase corp_2020': {29.0, 32.0},
 'city holding_2020': {61.0},
 'comerica inc_2019': {44.0},
 'cubesmart reit_2021': {19.0},
 'cytokinetics inc_2022': {24.0},
 'dampskibsselskabet norden_2019': {63.0},
 'dksh holding_2021': {17.0, 29.0, 73.0, 74.0},
 'evolution mining ltd_2020': {44.0},
 'fabege_2022': {45.0},
 'fastly inc_2018': {43.0},
 'fre

In [292]:
retrieval_files = list((RETRIEVAL_DIR/METHOD).rglob("*.json"))

results = []

for retrieval_file in retrieval_files:
    # if retrieval_file.stem.split("_")[0] != "amano corp":
    #     continue
    with open(retrieval_file, "r", encoding="utf-8") as f:
        rdata = json.load(f)
    
    
    retrieval_method = rdata.get("method")
    
    report_id = rdata.get("report_id")
    if report_id not in gold_pages_by_report:
        continue
    
    retrieved_results = rdata.get("items", [])
    retrieved_pages = set()
    
    for item in retrieved_results:
        metadata = item.get("metadata", {})
        page_numbers = metadata.get("page_no")
        retrieved_pages.add(page_numbers)
    
    gold_pages = gold_pages_by_report[report_id]
    
    # Evaluate retrieval results
    true_positives = len(retrieved_pages & gold_pages)
    false_positives = len(retrieved_pages - gold_pages)
    false_negatives = len(gold_pages - retrieved_pages)
        
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    missed_pages = sorted(gold_pages - retrieved_pages)

    missed_display_types = []
    for page in missed_pages:
        for display_type in display_types_by_page[(report_id, page)]:
            missed_display_types.append(str(display_type))
    
    results.append({
        "report_id": report_id,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "gold_pages": list(gold_pages),
        "retrieved_pages": list(retrieved_pages),
        "missed_pages": missed_pages,
        "annotated_display_type_missed_pages": missed_display_types,
    })
 
print(f"Retrieval Method: {retrieval_method}")
df = pd.DataFrame(results)


# if NO_GRAPHICS:
#     df.to_csv(f"{RESULTS_DIR}/{METHOD}_retrieval_eval_no_graphics.csv", index=False)
# else:
#     df.to_csv(f"{RESULTS_DIR}/{METHOD}_retrieval_eval.csv", index=False)
df

Retrieval Method: bm25


,report_id,precision,recall,f1_score,gold_pages,retrieved_pages,missed_pages,annotated_display_type_missed_pages
0,acuity brands inc_2022,0.4,1.0,0.571429,"[99.0, 101.0]","[99, 100, 101, 102, 28]",[],[]
1,addtech_2022,0.2,1.0,0.333333,[73.0],"[69, 70, 73, 74, 21]",[],[]
2,aixtron_2020,0.2,0.5,0.285714,"[50.0, 51.0]","[2, 43, 48, 81, 51]",[50.0],[Illustration]
3,allfunds group_2021,0.4,1.0,0.571429,"[125.0, 142.0]","[142, 151, 125, 126, 127]",[],[]
4,Allianz_2022,0.4,1.0,0.571429,"[93.0, 79.0]","[79, 49, 82, 24, 93]",[],[]
...,...,...,...,...,...,...,...,...
64,the simply good foods company_2018,0.2,1.0,0.333333,[91.0],"[114, 115, 90, 91, 92]",[],[]
65,toshiba tec corp_2022,0.2,1.0,0.333333,[31.0],"[13, 27, 28, 30, 31]",[],[]
66,uniper_2019,0.8,1.0,0.888889,"[16.0, 66.0, 67.0, 15.0]","[65, 66, 67, 15, 16]",[],[]
67,varta ag_2021,0.2,1.0,0.333333,[39.0],"[39, 40, 42, 24, 27]",[],[]


In [293]:
df["precision"].mean(), df["recall"].mean(), df["f1_score"].mean()

(np.float64(0.25797101449275356),
 np.float64(0.9746376811594203),
 np.float64(0.3927421210029906))